# HyraxNestedPandasDataset Example

This notebook demonstrates how to use `HyraxNestedPandasDataset` to load hierarchical data stored in nested-pandas format.

nested-pandas allows you to store structured data where each top-level row contains sub-dataframes as column values. This is useful for astronomical data with object-level metadata paired with multiple measurements per object (e.g., light curves, spectra, multiple observations).

## Setup: Create sample nested-pandas data

In [ ]:
import pandas as pd
import nested_pandas as npd
import tempfile
from pathlib import Path

# Create sample object catalog with metadata
catalog = pd.DataFrame({
    'object_id': ['source_001', 'source_002', 'source_003'],
    'ra': [10.5, 20.3, 15.1],
    'dec': [-5.2, -10.4, 5.6],
    'magnitude': [15.5, 16.2, 14.8],
    'classification': ['star', 'galaxy', 'star'],
})

# Create nested measurements dataframe with observations for each object
all_observations = [
    pd.DataFrame({
        'mjd': [58000.0, 58001.0, 58002.0],
        'flux': [100.5, 101.2, 99.8],
        'flux_err': [2.1, 2.0, 2.2],
        'filter': ['g', 'g', 'r'],
    }),
    pd.DataFrame({
        'mjd': [58000.5, 58001.5],
        'flux': [50.2, 51.5],
        'flux_err': [1.5, 1.6],
        'filter': ['i', 'i'],
    }),
    pd.DataFrame({
        'mjd': [58000.2, 58001.2, 58002.2, 58003.2],
        'flux': [200.1, 202.3, 199.8, 201.5],
        'flux_err': [3.2, 3.1, 3.3, 3.0],
        'filter': ['g', 'g', 'r', 'r'],
    }),
]

# Add nested observations to catalog
catalog['observations'] = all_observations

# Create nested-pandas NestedFrame
ndf = npd.NestedFrame(catalog)

print("Nested-pandas dataframe structure:")
print(ndf)
print()
print("First object's observations:")
print(ndf['observations'].iloc[0])

## Save and load with HyraxNestedPandasDataset

In [ ]:
from hyrax.datasets import HyraxNestedPandasDataset

# Save the nested dataframe to disk
with tempfile.TemporaryDirectory() as tmpdir:
    data_path = Path(tmpdir) / "sample_observations.pkl"
    ndf.to_pickle(data_path)
    
    # Create a HyraxNestedPandasDataset
    dataset = HyraxNestedPandasDataset(
        config={
            "data_set": {
                "HyraxNestedPandasDataset": {
                    "read_kwargs": {},
                }
            }
        },
        data_location=data_path,
    )
    
    print(f"Dataset length: {len(dataset)}")
    print()
    
    # Access top-level columns
    print("Top-level getters:")
    for idx in range(len(dataset)):
        obj_id = dataset.get_object_id(idx)
        ra = dataset.get_ra(idx)
        mag = dataset.get_magnitude(idx)
        print(f"  Index {idx}: {obj_id:12s} RA={ra:6.1f} Mag={mag:5.1f}")
    print()
    
    # Access nested columns
    print("Nested column getters:")
    mjds = dataset.get_observations_mjd(0)
    fluxes = dataset.get_observations_flux(0)
    filters = dataset.get_observations_filter(0)
    print(f"Object 0 observations:")
    for mjd, flux, filt in zip(mjds, fluxes, filters):
        print(f"    MJD={mjd:8.1f} Flux={flux:6.1f} Filter={filt}")

## Use in a Hyrax training configuration

In [ ]:
# Example data_request configuration for training
data_request = {
    "train": {
        "data": {
            "dataset_class": "HyraxNestedPandasDataset",
            "data_location": "/path/to/observations.pkl",
            "fields": [
                "object_id",      # Top-level field
                "magnitude",       # Top-level field
                "classification",  # Top-level field
                "observations.mjd",        # Nested field: MJD timestamps
                "observations.flux",       # Nested field: flux measurements
                "observations.flux_err",   # Nested field: flux uncertainties
            ],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
        },
    },
}

print("Example data_request configuration:")
import json
print(json.dumps(data_request, indent=2))

## Notes

### File formats
- **Pickle** (`.pkl`, `.pickle`): Recommended for nested-pandas dataframes since they contain Python objects that parquet cannot serialize.
- **Parquet** (`.parquet`): Supported for non-nested dataframes if they don't contain object columns.

### Column naming
- Top-level columns are accessed via `get_<column_name>()`
- Nested columns use dot notation in `fields`: `"nested_table.column_name"`
- Nested column getters are named `get_<nested_table>_<column>()`
- Column names are automatically sanitized: spaces and dashes become underscores

### Use cases
- **Light curves**: Multiple observations per object (star with variable brightness)
- **Spectroscopy**: Multiple spectra per source (different observations)
- **Multi-band imaging**: Different filter observations per object
- **Time series data**: Repeated measurements with timestamps